In [1]:
import random
random.seed(109)

In [2]:
class Mancala:
    # ------------------------ Helpers -------------------------- 

    # get the other player
    def _other(self, player):
        return 2 if player == 1 else 1

    # get the pit indices of the player vs the enemy 
    def _pit_indices(self, player):
        start, end = self.p1_pits_index if player == 1 else self.p2_pits_index
        return range(start, end + 1)

    # get the mancala index of the player vs the enemy 
    def _mancala_index(self, player):
        return self.p1_mancala_index if player == 1 else self.p2_mancala_index

    # Get the board index of the pit directly across from it 
    def _opposite_index(self, index):
        if index in self._pit_indices(1):
            offset = index - self.p1_pits_index[0]
            return self.p2_pits_index[1] - offset
        offset = index - self.p2_pits_index[0]
        return self.p1_pits_index[1] - offset

    # Get the winner 
    def get_winner(self):
        if not self.winning_eval():
            return None
        p1 = self.board[self.p1_mancala_index]
        p2 = self.board[self.p2_mancala_index]
        if p1 == p2:
            return 0
            
        return 1 if p1 > p2 else 2
    # ------------------------ Code ----------------------------------------- 
    def __init__(self, pits_per_player=6, stones_per_pit = 4):
        """
        The constructor for the Mancala class defines several instance variables:

        pits_per_player: This variable stores the number of pits each player has.
        stones_per_pit: It represents the number of stones each pit contains at the start of any game.
        board: This data structure is responsible for managing the Mancala board.
        current_player: This variable takes the value 1 or 2, as it's a two-player game, indicating which player's turn it is.
        moves: This is a list used to store the moves made by each player. It's structured in the format (current_player, chosen_pit).
        p1_pits_index: A list containing two elements representing the start and end indices of player 1's pits in the board data structure.
        p2_pits_index: Similar to p1_pits_index, it contains the start and end indices for player 2's pits on the board.
        p1_mancala_index and p2_mancala_index: These variables hold the indices of the Mancala pits on the board for players 1 and 2, respectively.
        """
        self.pits_per_player = pits_per_player
        self.board = [stones_per_pit] * ((pits_per_player+1) * 2)  # Initialize each pit with stones_per_pit number of stones 
        self.players = 2
        self.current_player = 1
        self.moves = []
        self.p1_pits_index = [0, self.pits_per_player-1]
        self.p1_mancala_index = self.pits_per_player
        self.p2_pits_index = [self.pits_per_player+1, len(self.board)-1-1]
        self.p2_mancala_index = len(self.board)-1
        
        # Zeroing the Mancala for both players
        self.board[self.p1_mancala_index] = 0
        self.board[self.p2_mancala_index] = 0

    def display_board(self):
        """
        Displays the board in a user-friendly format
        """
        player_1_pits = self.board[self.p1_pits_index[0]: self.p1_pits_index[1]+1]
        player_1_mancala = self.board[self.p1_mancala_index]
        player_2_pits = self.board[self.p2_pits_index[0]: self.p2_pits_index[1]+1]
        player_2_mancala = self.board[self.p2_mancala_index]

        print('P1               P2')
        print('     ____{}____     '.format(player_2_mancala))
        for i in range(self.pits_per_player):
            if i == self.pits_per_player - 1:
                print('{} -> |_{}_|_{}_| <- {}'.format(i+1, player_1_pits[i], 
                        player_2_pits[-(i+1)], self.pits_per_player - i))
            else:    
                print('{} -> | {} | {} | <- {}'.format(i+1, player_1_pits[i], 
                        player_2_pits[-(i+1)], self.pits_per_player - i))
            
        print('         {}         '.format(player_1_mancala))
        turn = 'P1' if self.current_player == 1 else 'P2'
        print('Turn: ' + turn)
        print()  # blank line between boards
        
    def valid_move(self, pit):
        """
        Function to check if the pit chosen by the current_player is a valid move.
        """

         # Must be a whole number
        if not isinstance(pit, int):
            return False

        # Check for real pit number (1 to pits_per_player)
        if pit < 1 or pit > self.pits_per_player:
            return False

        # Must have stones in it
        index = self._pit_indices(self.current_player)[pit - 1]
        return self.board[index] > 0
        
    def random_move_generator(self):
        """
        Function to generate random valid moves with non-empty pits for the random player
        """
        options = []
        # pit numbers 1..6
        for pit in range(1, self.pits_per_player + 1):  
            # check if non empty
            if self.valid_move(pit):                       
                options.append(pit)
    
        # no legal moves 
        if len(options) == 0:       
            return None
        return random.choice(options)
    
    def play(self, pit):
        """
        This function simulates a single move made by the current player using their selected pit (1-indexed).
        Do these checks in this order:
        1. If the game is already over (winning_eval() returns True), print "GAME OVER" and take no further action.
        2. Otherwise, if the chosen pit is not a valid move for the current player, print "INVALID MOVE" and take no action.
           The current player stays the same.
        3. Otherwise, distribute the stones according to the Mancala rules, including captures, and add
           (current_player, pit) to self.moves.

        Finally, apply the continuation rule: if the last stone landed in the current player's own mancala,
        the same player moves again; otherwise switch the current player so the other player takes their turn.
        If the move ends the game, there is no extra turn.

        Returns the board (self.board).
        """

        # check if game if over 
        if self.winning_eval():
            print("GAME OVER")
            return self.board
        # Check for valid move 
        if not self.valid_move(pit):
            print("INVALID MOVE")
            return self.board
            
        # Else distribute acoording to manacla moves 

        # get whose playing 
        player = self.current_player
        player_mancala = self._mancala_index(player)
        enemy_mancala = self._mancala_index(self._other(player))
        self.moves.append((player, pit))

        # 1) get all the stones from the chosen pit 
        index = self._pit_indices(player)[pit - 1]
        stones = self.board[index]
        self.board[index] = 0

        # 2) drop 1 stone per pit cc skipping opponents mancala 
        while stones > 0:
            index = (index + 1) % len(self.board)
            if index == enemy_mancala:
                continue
            self.board[index] += 1
            stones -= 1

        # 3) Capture (if u land in empty pit cature the stone and enemy pit stones) 
        if index in self._pit_indices(player) and self.board[index] == 1:
            opposite = self._opposite_index(index)
            self.board[player_mancala] += self.board[index] + self.board[opposite]
            self.board[index] = 0
            self.board[opposite] = 0

        # 4) check if this won the game 
        game_over = self.winning_eval()
        if game_over:
            winner = self.get_winner()
            if winner == 0:
                print("It's a tie!")
            else:
                print(f"Player {winner} wins!")

        # 5) continuation rule 
        if index == player_mancala and not game_over:
            return self.board
        self.current_player = self._other(player)
        return self.board

        # write your code here
        return self.board
    
    def winning_eval(self):
        """
        Function to verify if the game board has reached the winning state.
        Hint: If either of the players' pits are all empty, then it is considered a winning state.
        When the game is over, move any stones left on a player's side into that player's mancala.

        Returns True if the game is over and False otherwise (always a bool, never None or the winner).
        To report the winner, compare the two mancalas after winning_eval() returns True.
        """
        p1_empty = all(self.board[i] == 0 for i in self._pit_indices(1))
        p2_empty = all(self.board[i] == 0 for i in self._pit_indices(2))
        if not (p1_empty or p2_empty):
            return False

        for player in (1, 2):
            mancala = self._mancala_index(player)
            for i in self._pit_indices(player):
                self.board[mancala] += self.board[i]
                self.board[i] = 0
        return True     


In [3]:
# Mancala part 2
# Create a 6-pit, 4-stone game, then play 10 moves: you choose Player 1's moves,
# random_move_generator() chooses Player 2's. Display the board after every move.

game = Mancala(pits_per_player=6, stones_per_pit=4)
game.display_board()

# My moves for Player 1,
p1_moves = [3, 6, 2, 1, 6, 3, 2]
next_p1 = 0

# play the game for ten moves 
while len(game.moves) < 10 and not game.winning_eval():
    if game.current_player == 1:
        pit = p1_moves[next_p1]
        next_p1 += 1
        print(f"Player 1 chooses pit {pit}")
    else:
        pit = game.random_move_generator()
        print(f"Player 2 (random) chooses pit {pit}")
    game.play(pit)
    game.display_board()

# Results
print("List of moves:")
for player, pit in game.moves:
    print(f"Player {player} selected pit {pit}")

p1_score = game.board[game.p1_mancala_index]
p2_score = game.board[game.p2_mancala_index]
print(f"\nAfter {len(game.moves)} moves: Player 1 has {p1_score}, Player 2 has {p2_score}")
if game.winning_eval():
    winner = game.get_winner()
    print("Game over: it's a tie" if winner == 0 else f"Game over: Player {winner} wins")
elif p1_score != p2_score:
    print(f"Game not over yet. Player {1 if p1_score > p2_score else 2} is ahead.")
else:
    print("Game not over yet. The score is tied.")



P1               P2
     ____0____     
1 -> | 4 | 4 | <- 6
2 -> | 4 | 4 | <- 5
3 -> | 4 | 4 | <- 4
4 -> | 4 | 4 | <- 3
5 -> | 4 | 4 | <- 2
6 -> |_4_|_4_| <- 1
         0         
Turn: P1

Player 1 chooses pit 3
P1               P2
     ____0____     
1 -> | 4 | 4 | <- 6
2 -> | 4 | 4 | <- 5
3 -> | 0 | 4 | <- 4
4 -> | 5 | 4 | <- 3
5 -> | 5 | 4 | <- 2
6 -> |_5_|_4_| <- 1
         1         
Turn: P1

Player 1 chooses pit 6
P1               P2
     ____0____     
1 -> | 4 | 4 | <- 6
2 -> | 4 | 4 | <- 5
3 -> | 0 | 5 | <- 4
4 -> | 5 | 5 | <- 3
5 -> | 5 | 5 | <- 2
6 -> |_0_|_5_| <- 1
         2         
Turn: P2

Player 2 (random) chooses pit 3
P1               P2
     ____1____     
1 -> | 5 | 5 | <- 6
2 -> | 4 | 5 | <- 5
3 -> | 0 | 6 | <- 4
4 -> | 5 | 0 | <- 3
5 -> | 5 | 5 | <- 2
6 -> |_0_|_5_| <- 1
         2         
Turn: P1

Player 1 chooses pit 2
P1               P2
     ____1____     
1 -> | 5 | 5 | <- 6
2 -> | 0 | 5 | <- 5
3 -> | 1 | 6 | <- 4
4 -> | 6 | 0 | <- 3
5 -> | 6 | 5 | <- 2